In [1]:
# ============================================================
# FER2013 - ViT-B/16 Benchmark Experiment
# ============================================================

import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.cuda.amp import autocast, GradScaler

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "ViTB16"

TRAIN_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/train"
TEST_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/test"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 7
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
NUM_FOLDS = 5
RANDOM_SEED = 42

HEAD_ONLY_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = f"./{MODEL_NAME}_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)

# ============================================================
# TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# DATASET
# ============================================================

class FERDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.dataset = ImageFolder(root=root_dir)

        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        image, label = self.dataset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# LOAD DATASETS
# ============================================================

full_train_dataset = FERDataset(
    root_dir=TRAIN_DIR,
    transform=None
)

test_dataset = FERDataset(
    root_dir=TEST_DIR,
    transform=test_transform
)

class_names = full_train_dataset.dataset.classes

# ============================================================
# TARGETS + CLASS WEIGHTS
# ============================================================

targets = [label for _, label in full_train_dataset.dataset.samples]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(targets),
    y=targets
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(DEVICE)

# ============================================================
# DATASET WRAPPER
# ============================================================

class TransformSubset(Dataset):

    def __init__(self, subset, transform=None):

        self.subset = subset
        self.transform = transform

    def __len__(self):

        return len(self.subset)

    def __getitem__(self, idx):

        image, label = self.subset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# MODEL
# ============================================================

class ViTB16FER(nn.Module):

    def __init__(self, num_classes=7):

        super(ViTB16FER, self).__init__()

        self.backbone = models.vit_b_16(
            weights=models.ViT_B_16_Weights.IMAGENET1K_V1
        )

        feature_dim = self.backbone.heads.head.in_features

        # Replace classification head
        self.backbone.heads = nn.Sequential(

            nn.Dropout(0.4),

            nn.Linear(feature_dim, 256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        return self.backbone(x)

# ============================================================
# FREEZING STRATEGY
# ============================================================

def freeze_for_phase1(model):

    # Freeze entire backbone
    for param in model.backbone.parameters():
        param.requires_grad = False

    # Train classifier head only
    for param in model.backbone.heads.parameters():
        param.requires_grad = True

def unfreeze_for_phase2(model):

    # Freeze patch embedding layer
    for param in model.backbone.conv_proj.parameters():
        param.requires_grad = False

    # Freeze first encoder block
    for param in model.backbone.encoder.layers[0].parameters():
        param.requires_grad = False

    # Unfreeze remaining encoder blocks
    for block in model.backbone.encoder.layers[1:]:

        for param in block.parameters():
            param.requires_grad = True

    # Train classification head
    for param in model.backbone.heads.parameters():
        param.requires_grad = True

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred):

    return {

        "accuracy": accuracy_score(y_true, y_pred),

        "precision": precision_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        ),

        "per_class_f1": f1_score(
            y_true,
            y_pred,
            average=None,
            zero_division=0
        )
    }

# ============================================================
# PARAMETER COUNT
# ============================================================

def count_parameters(model):

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_params, trainable_params

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        with autocast(enabled=torch.cuda.is_available()):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_labels, all_preds)

    return epoch_loss, accuracy

# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(loader, leave=False):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with autocast(enabled=torch.cuda.is_available()):

                outputs = model(images)

                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_labels, all_preds)

    metrics = compute_metrics(all_labels, all_preds)

    return epoch_loss, accuracy, metrics, all_labels, all_preds

# ============================================================
# CROSS VALIDATION
# ============================================================

print("\n================================================")
print("Starting 5-Fold Stratified Cross Validation")
print("================================================\n")

fold_results = []

start_training_time = time.time()

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(targets)), targets)
):

    print(f"\n================ Fold {fold+1}/{NUM_FOLDS} ================\n")

    train_subset = Subset(full_train_dataset.dataset, train_idx)
    val_subset = Subset(full_train_dataset.dataset, val_idx)

    train_dataset = TransformSubset(
        train_subset,
        transform=train_transform
    )

    val_dataset = TransformSubset(
        val_subset,
        transform=test_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = ViTB16FER(
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    freeze_for_phase1(model)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )

    scaler = GradScaler()

    best_val_loss = np.inf
    best_model_wts = copy.deepcopy(model.state_dict())

    early_stop_counter = 0

    history = []

    for epoch in range(NUM_EPOCHS):

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")

        # ====================================================
        # PHASE 2
        # ====================================================

        if epoch == HEAD_ONLY_EPOCHS:

            print("\nUnfreezing deeper transformer layers...\n")

            unfreeze_for_phase2(model)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LEARNING_RATE
            )

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler
        )

        val_loss, val_acc, val_metrics, _, _ = validate(
            model,
            val_loader,
            criterion
        )

        scheduler.step(val_loss)

        history.append({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "val_loss": val_loss,

            "train_accuracy": train_acc,

            "val_accuracy": val_acc
        })

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_acc:.4f} | "
            f"Weighted F1: {val_metrics['weighted_f1']:.4f}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save(
                model.state_dict(),
                os.path.join(
                    OUTPUT_DIR,
                    f"{MODEL_NAME}_fold{fold+1}_best.pth"
                )
            )

            early_stop_counter = 0

        else:
            early_stop_counter += 1

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if early_stop_counter >= EARLY_STOPPING_PATIENCE:

            print("\nEarly stopping triggered.\n")

            break

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(best_model_wts)

    val_loss, val_acc, val_metrics, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    fold_results.append({

        "accuracy": val_metrics["accuracy"],

        "weighted_f1": val_metrics["weighted_f1"],

        "macro_f1": val_metrics["macro_f1"]
    })

    # ========================================================
    # SAVE TRAINING LOG
    # ========================================================

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{MODEL_NAME}_fold{fold+1}_training_log.csv"
        ),
        index=False
    )

# ============================================================
# CROSS VALIDATION SUMMARY
# ============================================================

cv_accuracies = [x["accuracy"] for x in fold_results]
cv_weighted_f1 = [x["weighted_f1"] for x in fold_results]
cv_macro_f1 = [x["macro_f1"] for x in fold_results]

mean_acc = np.mean(cv_accuracies)
std_acc = np.std(cv_accuracies)

mean_weighted_f1 = np.mean(cv_weighted_f1)
std_weighted_f1 = np.std(cv_weighted_f1)

mean_macro_f1 = np.mean(cv_macro_f1)
std_macro_f1 = np.std(cv_macro_f1)

# ============================================================
# FINAL TRAINING
# ============================================================

print("\n================================================")
print("Training Final Model on Full Training Dataset")
print("================================================\n")

final_train_dataset = FERDataset(
    root_dir=TRAIN_DIR,
    transform=train_transform
)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

final_model = ViTB16FER(
    num_classes=NUM_CLASSES
).to(DEVICE)

freeze_for_phase1(final_model)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

scaler = GradScaler()

best_model_wts = copy.deepcopy(final_model.state_dict())
best_loss = np.inf

history = []

for epoch in range(NUM_EPOCHS):

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")

    if epoch == HEAD_ONLY_EPOCHS:

        print("\nUnfreezing deeper transformer layers...\n")

        unfreeze_for_phase2(final_model)

        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, final_model.parameters()),
            lr=LEARNING_RATE
        )

    train_loss, train_acc = train_one_epoch(
        final_model,
        final_train_loader,
        criterion,
        optimizer,
        scaler
    )

    scheduler.step(train_loss)

    history.append({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "train_accuracy": train_acc
    })

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_acc:.4f}"
    )

    if train_loss < best_loss:

        best_loss = train_loss

        best_model_wts = copy.deepcopy(final_model.state_dict())

        torch.save(
            final_model.state_dict(),
            os.path.join(
                OUTPUT_DIR,
                f"{MODEL_NAME}_final_best.pth"
            )
        )

final_model.load_state_dict(best_model_wts)

# ============================================================
# TEST EVALUATION
# ============================================================

print("\n================================================")
print("Final Evaluation on Test Set")
print("================================================\n")

test_loss, test_acc, test_metrics, y_true, y_pred = validate(
    final_model,
    test_loader,
    criterion
)

# ============================================================
# FINAL RESULTS
# ============================================================

total_training_time = time.time() - start_training_time

total_params, trainable_params = count_parameters(
    final_model
)

print("\n================================================")
print("FINAL RESULTS")
print("================================================\n")

print(f"Mean CV Accuracy      : {mean_acc:.4f}")
print(f"Std CV Accuracy       : {std_acc:.4f}")

print(f"\nMean Weighted F1      : {mean_weighted_f1:.4f}")
print(f"Std Weighted F1       : {std_weighted_f1:.4f}")

print(f"\nMean Macro F1         : {mean_macro_f1:.4f}")
print(f"Std Macro F1          : {std_macro_f1:.4f}")

print("\n------------------------------------------------")

print(f"Final Test Accuracy   : {test_metrics['accuracy']:.4f}")

print(f"Final Weighted F1     : {test_metrics['weighted_f1']:.4f}")

print(f"Final Macro F1        : {test_metrics['macro_f1']:.4f}")

print("\n------------------------------------------------")

print(f"Total Parameters      : {total_params:,}")

print(f"Trainable Parameters  : {trainable_params:,}")

print(
    f"\nTotal Training Time   : "
    f"{total_training_time/60:.2f} minutes"
)

print("\n================================================")
print("Classification Report")
print("================================================\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

print("\n================================================")
print("Experiment Completed Successfully")
print("================================================")


Starting 5-Fold Stratified Cross Validation


================ Fold 1/5 ================

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 213MB/s]


Epoch [1/30]


Train Loss: 1.7395 | Val Loss: 1.5370 | Val Accuracy: 0.4425 | Weighted F1: 0.4328
Epoch [2/30]


Train Loss: 1.5669 | Val Loss: 1.4874 | Val Accuracy: 0.4434 | Weighted F1: 0.4492
Epoch [3/30]


...

Train Loss: 0.1613 | Train Accuracy: 0.9388
Epoch [28/30]


Train Loss: 0.1866 | Train Accuracy: 0.9293
Epoch [29/30]


Train Loss: 0.1835 | Train Accuracy: 0.9311
Epoch [30/30]


Train Loss: 0.1484 | Train Accuracy: 0.9446

Final Evaluation on Test Set




FINAL RESULTS

Mean CV Accuracy      : 0.6770
Std CV Accuracy       : 0.0080

Mean Weighted F1      : 0.6755
Std Weighted F1       : 0.0075

Mean Macro F1         : 0.6561
Std Macro F1          : 0.0093

------------------------------------------------
Final Test Accuracy   : 0.6966
Final Weighted F1     : 0.6954
Final Macro F1        : 0.6901

------------------------------------------------
Total Parameters      : 85,997,319
Trainable Parameters  : 78,165,255

Total Training Time   : 440.43 minutes

Classification Report

              precision    recall  f1-score   support

       angry     0.6870    0.5752    0.6261       958
   disgusted     0.7570    0.7297    0.7431       111
     fearful     0.5369    0.5547    0.5456      1024
       happy     0.8556    0.8952    0.8749      1774
     neutral     0.6342    0.6707    0.6520      1233
         sad     0.5812    0.5766    0.5789      1247
   surprised     0.8182    0.8014    0.8097       831

    accuracy                       